In [186]:
# This notebook involves cleaning and analyzing the data
import pandas as pd
import numpy as np

In [187]:
credits = pd.read_csv('../data/credits.csv')
movies = pd.read_csv('../data/movies.csv')

In [188]:
credits.head()

,movie_id,title,cast,crew
0,19995,Avatar,"[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."
1,285,Pirates of the Caribbean: At World's End,"[{""cast_id"": 4, ""character"": ""Captain Jack Spa...","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""de..."
2,206647,Spectre,"[{""cast_id"": 1, ""character"": ""James Bond"", ""cr...","[{""credit_id"": ""54805967c3a36829b5002c41"", ""de..."
3,49026,The Dark Knight Rises,"[{""cast_id"": 2, ""character"": ""Bruce Wayne / Ba...","[{""credit_id"": ""52fe4781c3a36847f81398c3"", ""de..."
4,49529,John Carter,"[{""cast_id"": 5, ""character"": ""John Carter"", ""c...","[{""credit_id"": ""52fe479ac3a36847f813eaa3"", ""de..."


In [189]:
movies.columns


Index(['budget', 'genres', 'homepage', 'id', 'keywords', 'original_language',
       'original_title', 'overview', 'popularity', 'production_companies',
       'production_countries', 'release_date', 'revenue', 'runtime',
       'spoken_languages', 'status', 'tagline', 'title', 'vote_average',
       'vote_count'],
      dtype='str')

In [190]:
credits.columns

Index(['movie_id', 'title', 'cast', 'crew'], dtype='str')

In [191]:
"""Here are the selected columns for recommendation model
    
    
"""
movies['id'].duplicated().any()

# Merge both 
movies = movies.merge(right=credits, left_on='id', right_on='movie_id')

In [192]:
movies.columns

Index(['budget', 'genres', 'homepage', 'id', 'keywords', 'original_language',
       'original_title', 'overview', 'popularity', 'production_companies',
       'production_countries', 'release_date', 'revenue', 'runtime',
       'spoken_languages', 'status', 'tagline', 'title_x', 'vote_average',
       'vote_count', 'movie_id', 'title_y', 'cast', 'crew'],
      dtype='str')

In [193]:
# Check whether dublicate title exists
movies['title_y'].duplicated().any()

np.True_

In [194]:
movies.rename(columns={
    'title_x': 'title'
    }, inplace=True
)

movies.columns

Index(['budget', 'genres', 'homepage', 'id', 'keywords', 'original_language',
       'original_title', 'overview', 'popularity', 'production_companies',
       'production_countries', 'release_date', 'revenue', 'runtime',
       'spoken_languages', 'status', 'tagline', 'title', 'vote_average',
       'vote_count', 'movie_id', 'title_y', 'cast', 'crew'],
      dtype='str')

In [195]:
# Extracting a release_year for each movie
movies['release_year'] = pd.to_datetime(
    movies['release_date']
).dt.year



In [196]:
# Checking Na values for year columnn
movies['release_year'].isna().sum()

np.int64(1)

In [197]:
movies[movies['release_year'].isna()]

,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,...,status,tagline,title,vote_average,vote_count,movie_id,title_y,cast,crew,release_year
4553,0,[],NaN,380097,[],en,America Is Still the Place,1971 post civil rights San Francisco seemed li...,0.0,[],...,Released,NaN,America Is Still the Place,0.0,0,380097,America Is Still the Place,[],[],NaN


In [198]:
# Drop the columns with na value
movies.dropna(subset=['release_year'], inplace=True)


In [199]:
movies['release_year'] = movies['release_year'].astype(int)
movies.columns


Index(['budget', 'genres', 'homepage', 'id', 'keywords', 'original_language',
       'original_title', 'overview', 'popularity', 'production_companies',
       'production_countries', 'release_date', 'revenue', 'runtime',
       'spoken_languages', 'status', 'tagline', 'title', 'vote_average',
       'vote_count', 'movie_id', 'title_y', 'cast', 'crew', 'release_year'],
      dtype='str')

In [200]:
movies = movies[
    [
        'id',
        'title',
        'overview',
        'genres',
        'keywords',
        'cast',
        'crew',
        'release_year'
    ]
]

In [201]:
# Checking which col has na values 
movies.isnull().sum()

id              0
title           0
overview        3
genres          0
keywords        0
cast            0
crew            0
release_year    0
dtype: int64

In [202]:
movies[movies['overview'].isna()]
movies.dropna(subset=['overview'], inplace=True)

In [203]:
movies.head()

,id,title,overview,genres,keywords,cast,crew,release_year
0,19995,Avatar,"In the 22nd century, a paraplegic Marine is di...","[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...","[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de...",2009
1,285,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...","[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...","[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...","[{""cast_id"": 4, ""character"": ""Captain Jack Spa...","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""de...",2007
2,206647,Spectre,A cryptic message from Bond’s past sends him o...,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[{""id"": 470, ""name"": ""spy""}, {""id"": 818, ""name...","[{""cast_id"": 1, ""character"": ""James Bond"", ""cr...","[{""credit_id"": ""54805967c3a36829b5002c41"", ""de...",2015
3,49026,The Dark Knight Rises,Following the death of District Attorney Harve...,"[{""id"": 28, ""name"": ""Action""}, {""id"": 80, ""nam...","[{""id"": 849, ""name"": ""dc comics""}, {""id"": 853,...","[{""cast_id"": 2, ""character"": ""Bruce Wayne / Ba...","[{""credit_id"": ""52fe4781c3a36847f81398c3"", ""de...",2012
4,49529,John Carter,"John Carter is a war-weary, former military ca...","[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[{""id"": 818, ""name"": ""based on novel""}, {""id"":...","[{""cast_id"": 5, ""character"": ""John Carter"", ""c...","[{""credit_id"": ""52fe479ac3a36847f813eaa3"", ""de...",2012


In [204]:
# Cleaning of the overview column
import re

def clean_overview(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)
    return text.split()

movies['overview'] = movies['overview'].apply(clean_overview)


In [205]:
import ast

def extract_names(text, key):
    items = ast.literal_eval(text)

    names = [] 

    for item in items:
        names.append(item[key])
    
    return names

In [206]:
movies['genres'] = movies['genres'].apply(
    lambda x: extract_names(x, "name")
)

movies['keywords'] = movies['keywords'].apply(
    lambda x: extract_names(x, "name")
)

In [207]:
# Extracting only useful 5 actors or cast
def extract_cast(text):
    items = ast.literal_eval(text)
    return [item['name'] for item in items[:5]]

In [208]:
movies['cast'] = movies['cast'].apply(lambda x: extract_cast(x))

In [209]:
movies[['cast', 'keywords', 'genres']]

,cast,keywords,genres
0,"[Sam Worthington, Zoe Saldana, Sigourney Weave...","[culture clash, future, space war, space colon...","[Action, Adventure, Fantasy, Science Fiction]"
1,"[Johnny Depp, Orlando Bloom, Keira Knightley, ...","[ocean, drug abuse, exotic island, east india ...","[Adventure, Fantasy, Action]"
2,"[Daniel Craig, Christoph Waltz, Léa Seydoux, R...","[spy, based on novel, secret agent, sequel, mi...","[Action, Adventure, Crime]"
3,"[Christian Bale, Michael Caine, Gary Oldman, A...","[dc comics, crime fighter, terrorist, secret i...","[Action, Crime, Drama, Thriller]"
4,"[Taylor Kitsch, Lynn Collins, Samantha Morton,...","[based on novel, mars, medallion, space travel...","[Action, Adventure, Science Fiction]"
...,...,...,...
4798,"[Carlos Gallardo, Jaime de Hoyos, Peter Marqua...","[united states–mexico barrier, legs, arms, pap...","[Action, Crime, Thriller]"
4799,"[Edward Burns, Kerry Bishé, Marsha Dietlein, C...",[],"[Comedy, Romance]"
4800,"[Eric Mabius, Kristin Booth, Crystal Lowe, Geo...","[date, love at first sight, narration, investi...","[Comedy, Drama, Romance, TV Movie]"
4801,"[Daniel Henney, Eliza Coupe, Bill Paxton, Alan...",[],[]


In [210]:
movies['crew'].iloc[0]



'[{"credit_id": "52fe48009251416c750aca23", "department": "Editing", "gender": 0, "id": 1721, "job": "Editor", "name": "Stephen E. Rivkin"}, {"credit_id": "539c47ecc3a36810e3001f87", "department": "Art", "gender": 2, "id": 496, "job": "Production Design", "name": "Rick Carter"}, {"credit_id": "54491c89c3a3680fb4001cf7", "department": "Sound", "gender": 0, "id": 900, "job": "Sound Designer", "name": "Christopher Boyes"}, {"credit_id": "54491cb70e0a267480001bd0", "department": "Sound", "gender": 0, "id": 900, "job": "Supervising Sound Editor", "name": "Christopher Boyes"}, {"credit_id": "539c4a4cc3a36810c9002101", "department": "Production", "gender": 1, "id": 1262, "job": "Casting", "name": "Mali Finn"}, {"credit_id": "5544ee3b925141499f0008fc", "department": "Sound", "gender": 2, "id": 1729, "job": "Original Music Composer", "name": "James Horner"}, {"credit_id": "52fe48009251416c750ac9c3", "department": "Directing", "gender": 2, "id": 2710, "job": "Director", "name": "James Cameron"},

In [211]:
# Extracting directors
def extract_director(text):
    
    for person in ast.literal_eval(text):
        if person['job'] == 'Director':
            return [person['name']]
    
    return []

In [212]:
movies['crew'] = movies['crew'].apply(extract_director)

In [213]:
movies.head()

,id,title,overview,genres,keywords,cast,crew,release_year
0,19995,Avatar,"[in, the, 22nd, century, a, paraplegic, marine...","[Action, Adventure, Fantasy, Science Fiction]","[culture clash, future, space war, space colon...","[Sam Worthington, Zoe Saldana, Sigourney Weave...",[James Cameron],2009
1,285,Pirates of the Caribbean: At World's End,"[captain, barbossa, long, believed, to, be, de...","[Adventure, Fantasy, Action]","[ocean, drug abuse, exotic island, east india ...","[Johnny Depp, Orlando Bloom, Keira Knightley, ...",[Gore Verbinski],2007
2,206647,Spectre,"[a, cryptic, message, from, bonds, past, sends...","[Action, Adventure, Crime]","[spy, based on novel, secret agent, sequel, mi...","[Daniel Craig, Christoph Waltz, Léa Seydoux, R...",[Sam Mendes],2015
3,49026,The Dark Knight Rises,"[following, the, death, of, district, attorney...","[Action, Crime, Drama, Thriller]","[dc comics, crime fighter, terrorist, secret i...","[Christian Bale, Michael Caine, Gary Oldman, A...",[Christopher Nolan],2012
4,49529,John Carter,"[john, carter, is, a, warweary, former, milita...","[Action, Adventure, Science Fiction]","[based on novel, mars, medallion, space travel...","[Taylor Kitsch, Lynn Collins, Samantha Morton,...",[Andrew Stanton],2012


In [214]:
# Remove spaces from colums 'genres, 'keywords', 'cast', 'crew'
def collapse(L):
    return [i.replace(" ", "") for i in L]


movies['genres'] = movies['genres'].apply(collapse)
movies['cast'] = movies['cast'].apply(collapse)
movies['cast'] = movies['cast'].apply(collapse)
movies['crew'] = movies['crew'].apply(collapse)

In [215]:
# Converting the year number to the following era
def get_era(year):
    if year < 1980:
        return ['classic']
    elif year < 2000:
        return ['modern']
    elif year < 2010:
        return ['millennium']
    elif year < 2020:
        return ['recent']
    else:
        return ['latest']

movies['era'] = movies['release_year'].apply(get_era)

# Tags for each movie
movies['tags'] = (
    movies['overview']
    + movies['genres']
    + movies['keywords']
    + movies['cast']
    + movies['crew']
    + movies['era']
)

In [216]:
# Dropping columns which are present in tags
movies.drop(columns=['overview', 'genres', 'keywords', 'cast', 'crew', 'era'], inplace=True)


In [218]:
new_df = movies[['id', 'title', 'release_year', 'tags']]

# Convert the tag to string 
new_df['tags'] = new_df['tags'].apply(lambda x: " ".join(x))

In [220]:
new_df['tags'][0]

'in the 22nd century a paraplegic marine is dispatched to the moon pandora on a unique mission but becomes torn between following orders and protecting an alien civilization Action Adventure Fantasy ScienceFiction culture clash future space war space colony society space travel futuristic romance space alien tribe alien planet cgi marine soldier battle love affair anti war power relations mind and soul 3d SamWorthington ZoeSaldana SigourneyWeaver StephenLang MichelleRodriguez JamesCameron millennium'

In [222]:
# Convert everything to lowercase
new_df['tags'] = new_df['tags'].apply(lambda x: x.lower())


In [225]:
# This does removes based on title mainly and if that removes 2 -3 movies we don't care really much about that
new_df.drop_duplicates(inplace=True)

In [228]:
new_df.isnull().sum()

id              0
title           0
release_year    0
tags            0
dtype: int64

In [230]:
# Saving the results 
new_df.to_csv('../data/processed_movies.csv', index=False)